# Assignment 4 — Part 2: Web Search (Inverted Index)

**Goal:** Build an inverted index over a collection of webpages and use it to answer search queries.

**Classes implemented:**
- `MySet` — Set with union/intersection
- `Position` — `(PageEntry, wordIndex)` tuple
- `WordEntry` — List of positions for one word across documents
- `PageIndex` — Per-page word → positions mapping (backed by `MyHashTable`)
- `PageEntry` — Reads a webpage file and builds its `PageIndex`
- `MyHashTable` — Custom hash table mapping word → `WordEntry`
- `InvertedPageIndex` — Global inverted index across all pages
- `SearchEngine` — Accepts `actions.txt` commands and prints answers

## Constants: Stop Words, Punctuation, Plural→Singular Map

In [1]:
import math
import re
import os

# Connector / stop words — counted for position but NOT stored in index
STOP_WORDS = {
    'a', 'an', 'the', 'they', 'these', 'this', 'for', 'is', 'are',
    'was', 'of', 'or', 'and', 'does', 'will', 'whose'
}

# Punctuation characters to replace with a space
PUNCTUATION = set('{}[]<>=().,;\'"?#!-:')

# Exhaustive plural → singular map (only the given pairs)
PLURAL_TO_SINGULAR = {
    'stacks':        'stack',
    'structures':    'structure',
    'applications':  'application',
}

def normalize(word):
    """
    Lowercase and map plural to singular where applicable.
    """
    w = word.lower()
    return PLURAL_TO_SINGULAR.get(w, w)

def tokenize(text):
    """
    Tokenize a line of webpage text:
      1. Replace each punctuation character with a space.
      2. Split on whitespace.
      3. Lowercase and normalize plurals.
    Returns list of (token, original_position_1_indexed) tuples
    where position counts ALL tokens (including stop words).
    """
    # Replace punctuation with spaces
    cleaned = ''.join(' ' if ch in PUNCTUATION else ch for ch in text)
    raw_tokens = cleaned.split()
    result = []
    for i, tok in enumerate(raw_tokens, start=1):
        result.append((normalize(tok), i))
    return result

print("Constants loaded.")
print("Example tokenize:", tokenize("Data structures is the study of structures for storing data."))

Constants loaded.
Example tokenize: [('data', 1), ('structure', 2), ('is', 3), ('the', 4), ('study', 5), ('of', 6), ('structure', 7), ('for', 8), ('storing', 9), ('data', 10)]


## Class: `MySet`

In [2]:
class MySet:
    """
    A simple set wrapper supporting addElement, union, and intersection.
    Internally backed by a Python set for O(1) membership.
    """

    def __init__(self):
        self._data = set()

    def addElement(self, element):
        """Add element to the set."""
        self._data.add(element)

    def union(self, otherSet):
        """Return a new MySet representing the union."""
        result = MySet()
        result._data = self._data | otherSet._data
        return result

    def intersection(self, otherSet):
        """Return a new MySet representing the intersection."""
        result = MySet()
        result._data = self._data & otherSet._data
        return result

    def __contains__(self, item):
        return item in self._data

    def __iter__(self):
        return iter(self._data)

    def __len__(self):
        return len(self._data)

    def __repr__(self):
        return f"MySet({self._data})"


# Quick test
s1 = MySet()
s1.addElement(1); s1.addElement(2)
s2 = MySet()
s2.addElement(2); s2.addElement(3)
print("Union:", s1.union(s2))
print("Intersection:", s1.intersection(s2))

Union: MySet({1, 2, 3})
Intersection: MySet({2})


## Class: `Position`

In [3]:
class Position:
    """
    Represents a tuple <PageEntry p, wordIndex i>.
    wordIndex is 1-based (position of the word in the document,
    counting ALL words including stop words).
    """

    def __init__(self, pageEntry, wordIndex: int):
        self._pageEntry = pageEntry   # PageEntry reference
        self._wordIndex = wordIndex   # 1-based position

    def getPageEntry(self):
        """Return the associated PageEntry."""
        return self._pageEntry

    def getWordIndex(self) -> int:
        """Return the 1-based word index within the page."""
        return self._wordIndex

    def __repr__(self):
        return f"Position({self._pageEntry.getPageName()}, {self._wordIndex})"

print("Position class defined.")

Position class defined.


## Class: `WordEntry`

In [4]:
class WordEntry:
    """
    Stores all Position entries for a single word across one or more documents.
    """

    def __init__(self, word: str):
        self._word = word
        self._positions = []   # list of Position objects

    def addPosition(self, position: Position):
        """Add a single position entry."""
        self._positions.append(position)

    def addPositions(self, positions: list):
        """Add multiple position entries at once."""
        self._positions.extend(positions)

    def getAllPositionsForThisWord(self) -> list:
        """Return the list of all Position entries for this word."""
        return list(self._positions)

    def getWord(self) -> str:
        return self._word

    def getTermFrequency(self, pageName: str) -> float:
        """
        Return the term frequency of this word in the given page:
            tf = (# occurrences in page) / (total words in page)
        """
        count = sum(1 for pos in self._positions
                    if pos.getPageEntry().getPageName() == pageName)
        page_entry = next((pos.getPageEntry() for pos in self._positions
                           if pos.getPageEntry().getPageName() == pageName), None)
        if page_entry is None or page_entry.getTotalWords() == 0:
            return 0.0
        return count / page_entry.getTotalWords()

    def __repr__(self):
        return f"WordEntry('{self._word}', {len(self._positions)} positions)"

print("WordEntry class defined.")

WordEntry class defined.


## Class: `MyHashTable`
Custom hash table mapping `word (str)` → `WordEntry`.

In [5]:
class MyHashTable:
    """
    Custom hash table used by PageIndex to map word → WordEntry.
    Uses separate chaining (list of buckets) for collision resolution.
    """

    DEFAULT_SIZE = 1024  # number of buckets

    def __init__(self, size: int = DEFAULT_SIZE):
        self._size = size
        self._buckets = [[] for _ in range(size)]  # each bucket: list of (key, WordEntry)
        self._count = 0

    def getHashIndex(self, s: str) -> int:
        """
        Hash function: polynomial rolling hash of the string.
        Maps word string → bucket index in [0, size).
        """
        h = 0
        for ch in s:
            h = (h * 31 + ord(ch)) % self._size
        return h

    def get(self, word: str):
        """Return the WordEntry for word, or None if not present."""
        idx = self.getHashIndex(word)
        for k, v in self._buckets[idx]:
            if k == word:
                return v
        return None

    def addPositionForWord(self, word: str, position: Position):
        """
        Add position to the WordEntry of word.
        Creates a new WordEntry if one does not exist yet.
        """
        idx = self.getHashIndex(word)
        for k, v in self._buckets[idx]:
            if k == word:
                v.addPosition(position)
                return
        # Not found — create new entry
        entry = WordEntry(word)
        entry.addPosition(position)
        self._buckets[idx].append((word, entry))
        self._count += 1

    def addPositionsForWord(self, wordEntry: WordEntry):
        """
        Merge a WordEntry into the table.
        If entry for the word already exists, merge positions;
        otherwise insert the new WordEntry.
        """
        word = wordEntry.getWord()
        idx = self.getHashIndex(word)
        for k, v in self._buckets[idx]:
            if k == word:
                v.addPositions(wordEntry.getAllPositionsForThisWord())
                return
        self._buckets[idx].append((word, wordEntry))
        self._count += 1

    def getWordEntries(self) -> list:
        """Return a flat list of all WordEntry objects in the table."""
        entries = []
        for bucket in self._buckets:
            for _, v in bucket:
                entries.append(v)
        return entries

    def __len__(self):
        return self._count

print("MyHashTable class defined.")

MyHashTable class defined.


## Class: `PageIndex`
Per-page index: word → WordEntry, backed by `MyHashTable`.

In [6]:
class PageIndex:
    """
    Stores one WordEntry for each unique (non-stop) word in a single document.
    Backed by MyHashTable.
    """

    def __init__(self):
        self._table = MyHashTable()

    def addPositionForWord(self, word: str, position: Position):
        """Delegate to hash table."""
        self._table.addPositionForWord(word, position)

    def getWordEntry(self, word: str):
        """Return WordEntry for word or None."""
        return self._table.get(word)

    def getWordEntries(self) -> list:
        """Return all WordEntry objects in this page index."""
        return self._table.getWordEntries()

    def getHashIndex(self, word: str) -> int:
        return self._table.getHashIndex(word)

print("PageIndex class defined.")

PageIndex class defined.


## Class: `PageEntry`
Reads a webpage file and builds its `PageIndex`.

In [7]:
class PageEntry:
    """
    Represents a single webpage.
    On construction, reads the file from `webpages/<pageName>`,
    tokenizes it, and builds a PageIndex.
    """

    WEBPAGES_DIR = "webpages"  # directory containing webpage files

    def __init__(self, pageName: str):
        self._pageName = pageName
        self._pageIndex = PageIndex()
        self._totalWords = 0   # total number of tokens (including stop words)
        self._buildIndex()

    def _buildIndex(self):
        """
        Read the page file, tokenize, and populate the PageIndex.
        Positions count ALL words (including stop words), but only
        non-stop-word tokens are stored in the index.
        """
        filepath = os.path.join(self.WEBPAGES_DIR, self._pageName)
        global_pos = 0  # running word position across all lines
        with open(filepath, 'r', encoding='utf-8', errors='replace') as f:
            for line in f:
                tokens = tokenize(line)
                for word, local_pos in tokens:
                    global_pos += 1
                    if word and word not in STOP_WORDS:
                        pos = Position(self, global_pos)
                        self._pageIndex.addPositionForWord(word, pos)
        self._totalWords = global_pos

    def getPageName(self) -> str:
        return self._pageName

    def getPageIndex(self) -> PageIndex:
        """Return the PageIndex for this webpage."""
        return self._pageIndex

    def getTotalWords(self) -> int:
        """Total word count (all tokens, including stop words)."""
        return self._totalWords

    def containsWord(self, word: str) -> bool:
        return self._pageIndex.getWordEntry(normalize(word)) is not None

    def getPositionsOfWord(self, word: str):
        """Return list of Position objects for word in this page (or empty list)."""
        entry = self._pageIndex.getWordEntry(normalize(word))
        return entry.getAllPositionsForThisWord() if entry else []

    def __repr__(self):
        return f"PageEntry('{self._pageName}')"

print("PageEntry class defined.")

PageEntry class defined.


## Class: `InvertedPageIndex`

In [8]:
class InvertedPageIndex:
    """
    Global inverted index over all added pages.
    Maps word → WordEntry containing positions across ALL pages.
    """

    def __init__(self):
        self._table = MyHashTable(size=4096)  # larger table for global index
        self._pages = {}   # pageName → PageEntry

    def addPage(self, pageEntry: PageEntry):
        """
        Add a PageEntry to the inverted index.
        Merges all word entries from the page into the global table.
        """
        self._pages[pageEntry.getPageName()] = pageEntry
        for wordEntry in pageEntry.getPageIndex().getWordEntries():
            self._table.addPositionsForWord(wordEntry)

    def getPagesWhichContainWord(self, word: str) -> MySet:
        """
        Return a MySet of PageEntry objects whose pages contain word.
        """
        result = MySet()
        w = normalize(word)
        entry = self._table.get(w)
        if entry:
            for pos in entry.getAllPositionsForThisWord():
                result.addElement(pos.getPageEntry())
        return result

    def getPageEntry(self, pageName: str):
        """Return the PageEntry by name, or None."""
        return self._pages.get(pageName)

    def getAllPages(self) -> dict:
        return dict(self._pages)

    def totalPageCount(self) -> int:
        return len(self._pages)

print("InvertedPageIndex class defined.")

InvertedPageIndex class defined.


## Class: `SearchEngine`
Parses and executes actions from `actions.txt`.

In [9]:
class SearchEngine:
    """
    Main search engine class.
    Maintains an InvertedPageIndex and handles action commands.
    
    Supported actions:
      addPage <x>                        — add webpage file x
      queryFindPagesWhichContainWord <x> — find all pages containing x
      queryFindPositionsOfWordInAPage <x> <y> — positions of word x in page y
    """

    def __init__(self):
        self._index = InvertedPageIndex()

    def performAction(self, actionMessage: str):
        """
        Parse and execute a single action string.
        Prints the result as specified in the assignment.
        """
        parts = actionMessage.strip().split()
        if not parts:
            return

        action = parts[0]

        # ── addPage ──────────────────────────────────────────────────────────
        if action == 'addPage':
            pageName = parts[1]
            page = PageEntry(pageName)
            self._index.addPage(page)

        # ── queryFindPagesWhichContainWord ───────────────────────────────────
        elif action == 'queryFindPagesWhichContainWord':
            word = normalize(parts[1])
            pages = self._index.getPagesWhichContainWord(word)
            if len(pages) == 0:
                print(f"No webpage contains word {parts[1]}")
            else:
                names = sorted(p.getPageName() for p in pages)
                print(','.join(names))

        # ── queryFindPositionsOfWordInAPage ──────────────────────────────────
        elif action == 'queryFindPositionsOfWordInAPage':
            word = normalize(parts[1])
            pageName = parts[2]
            page = self._index.getPageEntry(pageName)
            if page is None:
                print(f"No webpage {pageName} found")
            else:
                positions = page.getPositionsOfWord(word)
                if not positions:
                    print(f"Webpage {pageName} does not contain word {parts[1]}")
                else:
                    indices = sorted(pos.getWordIndex() for pos in positions)
                    print(','.join(map(str, indices)))

        else:
            print(f"[Unknown action: {action}]")

    def runActionsFile(self, actionsPath: str):
        """
        Read and execute all actions from a file, one per line.
        """
        with open(actionsPath, 'r') as f:
            for line in f:
                line = line.strip()
                if line:
                    self.performAction(line)

print("SearchEngine class defined.")

SearchEngine class defined.


## TF-IDF Scoring
Relevance scoring for multi-word queries.

In [10]:
def tfidf_score(word: str, pageEntry: PageEntry, index: InvertedPageIndex) -> float:
    """
    Compute TF-IDF relevance score for a word in a given page.

    TF  = (occurrences of word in page) / (total words in page)
    IDF = log(N / n_w)  where N = total pages, n_w = pages containing word

    Returns tf * idf.
    """
    w = normalize(word)
    N = index.totalPageCount()
    if N == 0:
        return 0.0

    # Term frequency
    positions = pageEntry.getPositionsOfWord(w)
    fw = len(positions)
    total_words = pageEntry.getTotalWords()
    tf = fw / total_words if total_words > 0 else 0.0

    # Inverse document frequency
    pages_with_word = index.getPagesWhichContainWord(w)
    n_w = len(pages_with_word)
    if n_w == 0:
        return 0.0
    idf = math.log(N / n_w)

    return tf * idf


def query(queryStr: str, index: InvertedPageIndex, top_n: int = 5):
    """
    Multi-word search query.
    Returns pages sorted by sum of TF-IDF scores for each query word.
    """
    words = [normalize(w) for w in queryStr.strip().split()
             if normalize(w) not in STOP_WORDS]

    # Collect all candidate pages (union of pages containing any query word)
    candidate_pages = MySet()
    for w in words:
        candidate_pages = candidate_pages.union(index.getPagesWhichContainWord(w))

    # Score each candidate page
    scored = []
    for page in candidate_pages:
        score = sum(tfidf_score(w, page, index) for w in words)
        scored.append((page.getPageName(), score))

    # Return top-n pages sorted by descending relevance
    scored.sort(key=lambda x: -x[1])
    return scored[:top_n]


print("TF-IDF scoring functions defined.")

TF-IDF scoring functions defined.


## Run: Execute `actions.txt` and Compare with `answers.txt`

In [11]:
# ── Configuration ─────────────────────────────────────────────────────────────
ACTIONS_FILE  = "datasets/Q2- webSearch/actions.txt"    # path to action commands file
ANSWERS_FILE  = "datasets/Q2- webSearch/answers.txt"    # path to expected answers file
WEBPAGES_DIR  = "datasets/Q2- webSearch/webpages"       # directory with webpage files

# Update the WEBPAGES_DIR used by PageEntry
PageEntry.WEBPAGES_DIR = WEBPAGES_DIR
# ─────────────────────────────────────────────────────────────────────────────

print("=" * 60)
print("Running actions from:", ACTIONS_FILE)
print("=" * 60)

import io, sys

# Capture printed output for comparison
captured_output = io.StringIO()
sys.stdout = captured_output

engine = SearchEngine()
engine.runActionsFile(ACTIONS_FILE)

sys.stdout = sys.__stdout__
output_lines = [l for l in captured_output.getvalue().splitlines() if l.strip()]

print("\n--- My Output ---")
for line in output_lines:
    print(line)

Running actions from: datasets/Q2- webSearch/actions.txt

--- My Output ---
No webpage contains word delhi
stack_datastructure_wiki
stack_datastructure_wiki
Webpage stack_datastructure_wiki does not contain word magazines
No webpage contains word allain
stack_cprogramming
stack_cprogramming
stack_cprogramming
stack_oracle
stack_cprogramming,stack_datastructure_wiki,stackoverflow
stackmagazine

--- Comparison: My Output vs Expected ---
#     Match    Expected                                                               Got                                     
----------------------------------------------------------------------------------------------------
1     ✓        No webpage contains word delhi                                         No webpage contains word delhi          
2     ✓        stack_datastructure_wiki                                               stack_datastructure_wiki                
3     ✓        stack_datastructure_wiki                                        

In [12]:
# Compare with expected answers
with open(ANSWERS_FILE, 'r') as f:
    expected_lines = [l.strip() for l in f if l.strip()]

print("\n--- Comparison: My Output vs Expected ---")
print(f"{'#':<5} {'Match':<8} {'Expected':<70} {'Got':<40}")
print("-" * 100)

correct = 0
for i, (got, exp) in enumerate(zip(output_lines, expected_lines), 1):
    match = "✓" if got == exp else "✗"
    if got == exp:
        correct += 1
    print(f"{i:<5} {match:<8} {exp:<70} {got:<40}")

total = max(len(output_lines), len(expected_lines))
print(f"\nScore: {correct}/{total} correct")


In [13]:
# Run a manual action
engine.performAction("queryFindPagesWhichContainWord data")
engine.performAction("queryFindPositionsOfWordInAPage structure page1.txt")

# Multi-word TF-IDF ranked query
results = query("data structures", engine._index)
for pageName, score in results:
    print(f"{pageName}: {score:.6f}")

print("Uncomment lines above to run manual queries.")